# 1. Setup

In [ ]:
import os, subprocess, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from PIL import Image
from scipy.ndimage import shift as ndi_shift, gaussian_filter, zoom as ndi_zoom
from scipy.signal import convolve2d
from skimage.registration import phase_cross_correlation
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr
import cv2
from scipy.signal import argrelextrema
from scipy.ndimage import gaussian_filter1d

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 8)})

In [ ]:
# IMAGES
IMAGE_DIR      = '/Users/shuyu/ENPH459-Super-Resolution/imaging/sr/images'
CENTER_PATH    = f'{IMAGE_DIR}/center.png' # XPR off
SHIFT_PATHS    = [f'{IMAGE_DIR}/shift_{i}.png' for i in range(4)] # 4 XPR positions

# CALIBRATIONS
CAL_DIR        = '/Users/shuyu/ENPH459-Super-Resolution/imaging/sr/cal'
PSF_DATA_PATH  = f'{CAL_DIR}/psf_mtf_results_data_pos0.npz'
ISO_CHART_PATH = f'{CAL_DIR}/ISO_12233-reschart.pdf'

# SPECS
PIXEL_PITCH_UM = 3.45 # um / pixel
UPSAMPLE_FACTOR = 2
f = UPSAMPLE_FACTOR
PITCH_LR_MM = PIXEL_PITCH_UM * 1e-3
PITCH_HR_MM = PITCH_LR_MM / f
NYQUIST_LR  = 0.5 / PITCH_LR_MM   # cycles/mm
NYQUIST_HR  = 0.5 / PITCH_HR_MM
NOMINAL_SHIFTS = [
    (-0.5, -0.5), # shift_0 (top-left)
    (+0.5, -0.5), # shift_1 (bottom-left)
    (-0.5, +0.5), # shift_2 (top-right)
    (+0.5, +0.5), # shift_3 (bottom-right)
]
sub_shifts = NOMINAL_SHIFTS # Could also use phase-correlation estimates instead

# COLORS for plotting
COLORS = {'Native': 'C0', 'SAA': 'C2', 'SAA + IBP': 'C3', 'TV-SR': 'C4'}

In [ ]:
def load_gray(path):
    """Load any image as float64 grayscale."""
    img = np.array(Image.open(path), dtype=np.float64)
    return img.mean(axis=2) if img.ndim == 3 else img

def hr_crop(img, cr, cc, crop_r, factor):
    """Extract a crop from an HR image using LR coordinates."""
    return img[(cr-crop_r)*factor:(cr+crop_r)*factor,
               (cc-crop_r)*factor:(cc+crop_r)*factor]

def blur(img, kernel):
    """Convolve image with a PSF kernel (symmetric boundary)."""
    return convolve2d(img, kernel, mode='same', boundary='symm')

def forward_model(hr, kernel, shift_yx, factor):
    """
    Simulate a LR observation from an HR image.
    
    Pipeline: HR → blur(PSF) → sub-pixel shift → decimate
    
    Parameters
    ----------
    hr        : 2D HR image
    kernel    : PSF at HR resolution
    shift_yx  : (dy, dx) shift in LR pixels
    factor    : decimation factor (2 for 2× SR)
    """
    blurred = blur(hr, kernel)
    shifted = ndi_shift(blurred,
                        (shift_yx[0] * factor, shift_yx[1] * factor),
                        order=3, mode='nearest')
    return shifted[::factor, ::factor]  # decimate

def back_project(error_lr, kernel, shift_yx, factor, hr_shape):
    """
    Adjoint of the forward model: project a LR error back onto the HR grid.
    
    Pipeline: upsample (zero-insert) → reverse shift → correlate with PSF
    """
    h_hr, w_hr = hr_shape
    # Zero-insert upsample
    up = np.zeros((error_lr.shape[0] * factor, error_lr.shape[1] * factor))
    up[::factor, ::factor] = error_lr
    # Crop / pad to HR size
    up = up[:h_hr, :w_hr] if up.shape[0] >= h_hr else \
         np.pad(up, ((0, h_hr - up.shape[0]), (0, w_hr - up.shape[1])))
    # Reverse shift
    shifted = ndi_shift(up,
                        (-shift_yx[0] * factor, -shift_yx[1] * factor),
                        order=3, mode='nearest')
    # Correlation with PSF = convolution with flipped kernel (adjoint)
    return blur(shifted, kernel[::-1, ::-1])

In [ ]:
# Load all images
lr_ref = load_gray(CENTER_PATH)
lr_images = [load_gray(p) for p in SHIFT_PATHS]

print(f'Reference image : {lr_ref.shape}  (center, XPR off)')
print(f'Shifted images  : {len(lr_images)} × {lr_images[0].shape}')
print(f'Dynamic range   : [{lr_ref.min():.0f}, {lr_ref.max():.0f}]')

In [ ]:
EDGE_ROI_LR     = (550, 720, 580, 590)  # (r0, r1, c0, c1)
EDGE_ORIENT     = 'vertical'            # 'vertical' or 'horizontal' edge
STAR_CENTER_LR  = (804, 978)           # (row, col)
STAR_ROI_R      = 360

# Radial ring analysis parameters
RING_ANGLE_MIN = 230   # degrees
RING_ANGLE_MAX = 30    # degrees (wraps around 0)
RING_R_MIN = 60        # ignore uniform centre
RING_R_MAX = 360       # stop before black square border

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.imshow(lr_ref, cmap='gray')
r0, r1, c0, c1 = EDGE_ROI_LR
ax.add_patch(Rectangle((c0, r0), c1-c0, r1-r0, lw=2, edgecolor='cyan', facecolor='none'))
ax.set_title('Edge ROI')
ax.axis('off')

ax = axes[1]
ax.imshow(lr_ref, cmap='gray')
cy, cx = STAR_CENTER_LR
t = np.linspace(0, 2*np.pi, 200)
ax.plot(cx + STAR_ROI_R*np.cos(t), cy + STAR_ROI_R*np.sin(t), 'r-', lw=1)
ax.plot(cx, cy, 'r+', markersize=12, mew=2)

# Plot radial lines for ring analysis wedge
for angle_deg in [RING_ANGLE_MIN, RING_ANGLE_MAX]:
    angle_rad = np.radians(angle_deg)
    x_end = cx + STAR_ROI_R * np.cos(angle_rad)
    y_end = cy + STAR_ROI_R * np.sin(angle_rad)
    ax.plot([cx, x_end], [cy, y_end], 'r--', lw=1, alpha=0.7)

ax.set_title('Star ROI')
ax.axis('off')

plt.tight_layout(); plt.show()

In [ ]:
PSF_HALFWIDTH = 3

psf_data = np.load(PSF_DATA_PATH, allow_pickle=True)
psf_full = psf_data['psf_avg']
cy, cx = psf_full.shape[0] // 2, psf_full.shape[1] // 2
psf_kernel = psf_full[cy - PSF_HALFWIDTH : cy + PSF_HALFWIDTH + 1,
                        cx - PSF_HALFWIDTH : cx + PSF_HALFWIDTH + 1].copy()
psf_kernel = np.clip(psf_kernel, 0, None)
psf_kernel /= psf_kernel.sum()
psf_hr = ndi_zoom(psf_kernel, f, order=3)
psf_hr = np.clip(psf_hr, 0, None)
psf_hr /= psf_hr.sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.imshow(psf_kernel, cmap='inferno', interpolation='nearest')
ax.set_title(f'PSF kernel {psf_kernel.shape}')
ax.axis('off')

ax = axes[1]
freq = psf_data['freq']
mtf_mean = psf_data['mtf_mean']
mtf_std = psf_data['mtf_std']
ax.plot(freq, mtf_mean, 'b-', lw=1.5)
ax.fill_between(freq, mtf_mean - mtf_std, mtf_mean + mtf_std, alpha=0.3)
ax.set_xlabel('Frequency (cy/mm)')
ax.set_ylabel('MTF')
ax.set_title(f"MTF (MTF50={psf_data['mtf50']:.2f})")
ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# 2. SAA

In [ ]:
def shift_and_add(lr_list, shifts_yx, factor=2, order=3):
    """
    Returns the SAA super-resolved image after upsampling each LR frame, shifting, and averaging.
    """
    h_lr, w_lr = lr_list[0].shape
    accumulator = np.zeros((h_lr * factor, w_lr * factor))
    for lr, (dy, dx) in zip(lr_list, shifts_yx):
        upsampled = ndi_zoom(lr, factor, order=order)
        shifted = ndi_shift(upsampled,
                            (dy * factor, dx * factor),
                            order=3, mode='nearest')
        accumulator += shifted
    return accumulator / len(lr_list)

baseline_bicubic = ndi_zoom(lr_ref, f, order=3) # just upscaling the center image, no SR)
sr_saa_nearest = shift_and_add(lr_images, sub_shifts, f, order=0)
sr_saa_bicubic = shift_and_add(lr_images, sub_shifts, f, order=3)

In [ ]:
cr, cc = STAR_CENTER_LR
crop_r = STAR_ROI_R

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, img, title in zip(axes,
    [ndi_zoom(lr_ref, f, order=0), baseline_bicubic, sr_saa_nearest, sr_saa_bicubic],
    ['Native (2×)', 'Bicubic 2×', 'SAA nearest 2×', 'SAA bicubic 2×']):
    ax.imshow(hr_crop(img, cr, cc, crop_r, f), cmap='gray', interpolation='nearest')
    ax.set_title(title); ax.axis('off')
plt.suptitle('SAA comparison (centre crop)', fontsize=12)
plt.tight_layout(); plt.show()

# 3. SAA + IBP

In [ ]:
IBP_ITERATIONS  = 20
IBP_STEP_SIZE   = 0.5

In [ ]:
def ibp(lr_list, shifts_yx, kernel, hr_init, factor=2,
        n_iter=10, step=0.5, verbose=True):
    """
    Returns the final HR estimate and the per-iteration MSE errors for convergence monitoring.
    """
    hr = hr_init.copy()
    n = len(lr_list)
    errors = []

    for it in range(n_iter):
        correction = np.zeros_like(hr)
        total_err = 0.0

        for lr, s in zip(lr_list, shifts_yx):
            sim = forward_model(hr, kernel, s, factor)
            mh, mw = min(sim.shape[0], lr.shape[0]), min(sim.shape[1], lr.shape[1])
            err = lr[:mh, :mw] - sim[:mh, :mw]
            total_err += np.mean(err ** 2)
            correction += back_project(err, kernel, s, factor, hr.shape)

        hr += step * correction / n
        hr = np.clip(hr, 0, 255)
        errors.append(total_err / n)

        if verbose and (it % 5 == 0 or it == n_iter - 1):
            print(f'  iter {it:3d}/{n_iter}  MSE = {errors[-1]:.3f}')

    return hr, errors

In [ ]:
print(f'Running IBP  (iterations={IBP_ITERATIONS}, step={IBP_STEP_SIZE}) ...')
sr_ibp, ibp_errors = ibp(
    lr_images, sub_shifts, psf_hr, sr_saa_bicubic.copy(),
    factor=f, n_iter=IBP_ITERATIONS, step=IBP_STEP_SIZE
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ax = axes[0]
ax.plot(ibp_errors, 'C2-', lw=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('MSE')
ax.set_title('IBP convergence'); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.imshow(hr_crop(baseline_bicubic, cr, cc, crop_r, f), cmap='gray', interpolation='nearest')
ax.set_title('Bicubic baseline'); ax.axis('off')

ax = axes[2]
ax.imshow(hr_crop(sr_ibp, cr, cc, crop_r, f), cmap='gray', interpolation='nearest')
ax.set_title('IBP result'); ax.axis('off')
plt.tight_layout(); plt.show()

# 4. TV SR

In [ ]:
TV_ITERATIONS   = 20
TV_STEP_SIZE    = 0.15
TV_LAMBDA       = 0.005

In [ ]:
def tv_gradient(img):
    eps = 1e-8
    dy = np.zeros_like(img); dy[:-1, :] = img[1:, :] - img[:-1, :]
    dx = np.zeros_like(img); dx[:, :-1] = img[:, 1:] - img[:, :-1]
    mag = np.sqrt(dy**2 + dx**2 + eps)
    ny, nx = dy / mag, dx / mag
    div = np.zeros_like(img)
    div[1:, :]  -= ny[:-1, :]; div[:-1, :] += ny[:-1, :]
    div[:, 1:]  -= nx[:, :-1]; div[:, :-1] += nx[:, :-1]
    return -div


def tv_sr(lr_list, shifts_yx, kernel, hr_init, factor=2,
          n_iter=50, step=0.1, lam=0.01, verbose=True):
    """
    Returns final HR estimate and per-iteration MSE errors for convergence monitoring.
    """
    hr = hr_init.copy()
    n = len(lr_list)
    errors = []

    for it in range(n_iter):
        data_grad = np.zeros_like(hr)
        total_err = 0.0

        for lr, s in zip(lr_list, shifts_yx):
            sim = forward_model(hr, kernel, s, factor)
            mh, mw = min(sim.shape[0], lr.shape[0]), min(sim.shape[1], lr.shape[1])
            residual = sim[:mh, :mw] - lr[:mh, :mw]
            total_err += np.mean(residual ** 2)
            data_grad += back_project(residual, kernel, s, factor, hr.shape)

        data_grad /= n
        hr -= step * (data_grad + lam * tv_gradient(hr))
        hr = np.clip(hr, 0, 255)
        errors.append(total_err / n)

        if verbose and (it % 10 == 0 or it == n_iter - 1):
            print(f'  iter {it:3d}/{n_iter}  MSE = {errors[-1]:.3f}')

    return hr, errors

In [ ]:
print(f'Running TV-SR (iterations={TV_ITERATIONS}, step={TV_STEP_SIZE}, lambda={TV_LAMBDA}) ...')
sr_tv, tv_errors = tv_sr(
    lr_images, sub_shifts, psf_hr, sr_saa_bicubic.copy(),
    factor=f, n_iter=TV_ITERATIONS, step=TV_STEP_SIZE, lam=TV_LAMBDA
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ax = axes[0]
ax.plot(tv_errors, 'C3-', lw=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('MSE')
ax.set_title('TV-SR convergence'); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.imshow(hr_crop(baseline_bicubic, cr, cc, crop_r, f), cmap='gray', interpolation='nearest')
ax.set_title('Bicubic baseline'); ax.axis('off')

ax = axes[2]
ax.imshow(hr_crop(sr_tv, cr, cc, crop_r, f), cmap='gray', interpolation='nearest')
ax.set_title('TV-SR result'); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Collect and compare all results
results = {
    'Native':     ndi_zoom(lr_ref, f, order=0),  # nearest-neighbour (no new info)
    'Bicubic 2×': baseline_bicubic,               # interpolation only
    'SAA':        sr_saa_bicubic,
    'SAA + IBP':  sr_ibp,
    'TV-SR':      sr_tv,
}

# Side-by-side
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
for ax, (name, img) in zip(axes, results.items()):
    ax.imshow(hr_crop(img, cr, cc, crop_r, f), cmap='gray', interpolation='nearest')
    ax.set_title(name, fontsize=10); ax.axis('off')
plt.suptitle('All methods — centre of ISO 12233 chart', fontsize=13)
plt.tight_layout(); plt.show()

# 6. Slanted Edge Measurement 

In [ ]:
def slanted_edge_mtf(img, roi, orientation='vertical', oversample=4):
    r0, r1, c0, c1 = roi
    patch = img[r0:r1, c0:c1].copy()
    if orientation == 'horizontal':
        patch = patch.T
    rows, cols = patch.shape

    edge_pts = []
    for r in range(rows):
        d = np.abs(np.diff(patch[r, :].astype(float)))
        if d.max() < 5:
            continue
        peaks = np.where(d > d.max() * 0.5)[0]
        if len(peaks) > 0:
            edge_pts.append((r, np.average(peaks, weights=d[peaks])))

    if len(edge_pts) < 10:
        return None, None, None, None

    edge_pts = np.array(edge_pts)
    slope, intercept = np.polyfit(edge_pts[:, 0], edge_pts[:, 1], 1)
    cos_a = np.cos(np.arctan(slope))

    dists, vals = [], []
    for r in range(rows):
        edge_c = slope * r + intercept
        for c in range(cols):
            dists.append((c - edge_c) * cos_a)
            vals.append(patch[r, c])

    dists, vals = np.array(dists), np.array(vals)
    bw = 1.0 / oversample
    bins = np.arange(dists.min(), dists.max(), bw)
    esf = np.zeros(len(bins))
    cnt = np.zeros(len(bins))
    idx = np.clip(((dists - dists.min()) / bw).astype(int), 0, len(bins) - 1)
    for i in range(len(dists)):
        esf[idx[i]] += vals[i]; cnt[idx[i]] += 1
    valid = cnt > 0
    esf[valid] /= cnt[valid]
    esf[~valid] = np.interp(bins[~valid], bins[valid], esf[valid])

    lsf = np.diff(gaussian_filter(esf, sigma=0.5))
    lsf_w = lsf * np.hanning(len(lsf))
    mtf = np.abs(np.fft.fft(lsf_w))
    mtf = mtf[:len(mtf) // 2]
    mtf /= mtf[0]
    freq = np.arange(len(mtf)) / (len(mtf) * bw * 2)

    return freq, mtf, esf, lsf

In [ ]:
roi_hr = tuple(x * f for x in EDGE_ROI_LR)

mtf_curves = {}
for img, name, pitch in [(lr_ref, 'Native', PITCH_LR_MM)] + [(results[n], n, PITCH_HR_MM) for n in ['SAA', 'SAA + IBP', 'TV-SR']]:
    roi = EDGE_ROI_LR if name == 'Native' else roi_hr
    fr, mt, _, _ = slanted_edge_mtf(img, roi, EDGE_ORIENT)
    if fr is not None:
        mtf_curves[name] = (fr / pitch, mt)

fig, axes = plt.subplots(1, 1, figsize=(9, 5))
ax = axes
for name, (freq, mtf) in mtf_curves.items():
    ax.plot(freq, mtf, color=COLORS.get(name, 'gray'), lw=1.5, label=name)

ax.axvline(NYQUIST_LR, ls='--', color='gray', alpha=0.6, label=f'LR Nyquist ({NYQUIST_LR:.0f} cy/mm)')
ax.axvline(NYQUIST_HR, ls=':', color='gray', alpha=0.6, label=f'HR Nyquist ({NYQUIST_HR:.0f} cy/mm)')
ax.axhline(0.5, ls='--', color='orange', alpha=0.4)

ax.set_xlabel('Spatial frequency (cycles/mm)')
ax.set_ylabel('MTF')
ax.set_title('Slanted-Edge MTF')
ax.set_xlim(0, NYQUIST_HR)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 7. Radial Contrast Measurement

In [ ]:
def get_ring_angles(angle_min, angle_max, step=3):
    if angle_min > angle_max:
        return list(range(angle_min, 360, step)) + list(range(0, angle_max + 1, step))
    return list(range(angle_min, angle_max + 1, step))

def radial_profile(img, center_rc, theta_rad, max_r):
    cy, cx = center_rc
    prof = []
    for r in range(max_r):
        y = int(round(cy + r * np.sin(theta_rad)))
        x = int(round(cx + r * np.cos(theta_rad)))
        if 0 <= y < img.shape[0] and 0 <= x < img.shape[1]:
            prof.append(float(img[y, x]))
        else:
            prof.append(np.nan)
    return np.array(prof)

def find_ring_pairs(profile, r_min, r_max, smooth_sigma=1.0, peak_order=3):
    prof_s = gaussian_filter1d(profile, sigma=smooth_sigma)
    peaks = argrelextrema(prof_s, np.greater, order=peak_order)[0]
    valleys = argrelextrema(prof_s, np.less, order=peak_order)[0]
    peaks = peaks[(peaks >= r_min) & (peaks < r_max)]
    valleys = valleys[(valleys >= r_min) & (valleys < r_max)]
    extrema = sorted([(p, prof_s[p], 'P') for p in peaks] + [(v, prof_s[v], 'V') for v in valleys], key=lambda x: x[0])
    pairs = []
    for i in range(len(extrema) - 1):
        r1, v1, t1 = extrema[i]
        r2, v2, t2 = extrema[i + 1]
        if t1 != t2:
            hi, lo = max(v1, v2), min(v1, v2)
            if (hi + lo) > 10 and (hi - lo) > 5:
                pairs.append(((r1 + r2) / 2, (hi - lo) / (hi + lo), abs(r2 - r1)))
    return pairs, extrema

def radial_ring_contrast(img, center_rc, r_min=50, r_max=140, angle_min=330, angle_max=40, smooth_sigma=1.0, peak_order=3):
    degs = get_ring_angles(angle_min, angle_max)
    max_r = r_max + 5
    all_profiles, all_pairs = [], []
    for d in degs:
        theta = np.radians(d)
        prof = radial_profile(img, center_rc, theta, max_r)
        all_profiles.append(prof)
        if not np.any(np.isnan(prof[r_min:r_max])):
            pairs, _ = find_ring_pairs(prof, r_min, r_max, smooth_sigma, peak_order)
            all_pairs.extend(pairs)
    avg_profile = np.nanmean(all_profiles, axis=0)
    if not all_pairs:
        return np.array([]), np.array([]), np.array([]), avg_profile
    all_pairs = np.array(all_pairs)
    pitch_mm = PIXEL_PITCH_UM * 1e-3
    radii, contrasts, freq_cymm = all_pairs[:, 0], all_pairs[:, 1], 1.0 / (2 * all_pairs[:, 2] * pitch_mm)
    return radii, contrasts, freq_cymm, avg_profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cy, cx = STAR_CENTER_LR

ring_data = {}
star_hr = (cy * f, cx * f)
for name, img, center, r_min, r_max, peak_ord in [
    ('Native', lr_ref, STAR_CENTER_LR, RING_R_MIN, RING_R_MAX, 3),
    ('SAA', results['SAA'], star_hr, RING_R_MIN * f, RING_R_MAX * f, 4),
    ('SAA + IBP', results['SAA + IBP'], star_hr, RING_R_MIN * f, RING_R_MAX * f, 4),
    ('TV-SR', results['TV-SR'], star_hr, RING_R_MIN * f, RING_R_MAX * f, 4),
]:
    r, c, f_vals, _ = radial_ring_contrast(img, center, r_min, r_max, RING_ANGLE_MIN, RING_ANGLE_MAX, peak_order=peak_ord)
    ring_data[name] = {'radii': r / f if name != 'Native' else r, 'contrast': c, 'freq': f_vals}

def plot_binned(ax, x, y, color, label, bin_edges):
    bc, bm = [], []
    for j in range(len(bin_edges) - 1):
        mask = (x >= bin_edges[j]) & (x < bin_edges[j + 1])
        if mask.sum() >= 2:
            bc.append((bin_edges[j] + bin_edges[j + 1]) / 2)
            bm.append(np.mean(y[mask]))
    if bc:
        ax.plot(bc, bm, 'o-', color=color, lw=1.5, markersize=5, label=label)

ax = axes[0]
show_angles = get_ring_angles(RING_ANGLE_MIN, RING_ANGLE_MAX, step=8)
cmap = plt.cm.viridis(np.linspace(0, 1, len(show_angles)))
for i, d in enumerate(show_angles):
    theta = np.radians(d)
    prof = radial_profile(lr_ref, STAR_CENTER_LR, theta, RING_R_MAX + 5)
    ax.plot(range(len(prof)), prof, color=cmap[i], alpha=0.4, lw=0.6)
_, _, _, avg_prof = radial_ring_contrast(lr_ref, STAR_CENTER_LR, RING_R_MIN, RING_R_MAX, RING_ANGLE_MIN, RING_ANGLE_MAX)
ax.plot(range(len(avg_prof)), avg_prof, 'k-', lw=2, label='Mean profile')
ax.axvline(RING_R_MIN, ls=':', color='cyan', alpha=0.7, label=f'r_min={RING_R_MIN}')
ax.axvline(RING_R_MAX, ls=':', color='magenta', alpha=0.7, label=f'r_max={RING_R_MAX}')
ax.set_xlabel('Radius from centre (px)')
ax.set_ylabel('Intensity')
ax.set_title('Radial intensity profiles (native)')
ax.set_xlim(0, RING_R_MAX + 5)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[1]
for name, data in ring_data.items():
    freq, contrast = data['freq'], data['contrast']
    if len(freq) == 0: continue
    col = COLORS.get(name, 'gray')
    ax.scatter(freq, contrast, s=10, alpha=0.3, color=col)
    bin_edges = np.arange(0, max(freq) + 5, 5)
    plot_binned(ax, freq, contrast, col, name, bin_edges)

ax.axhline(0.1, ls='--', color='orange', alpha=0.5, label='Contrast = 0.1')
ax.set_xlabel('Spatial frequency (cycles/mm)')
ax.set_ylabel('Michelson contrast')
ax.set_title('Ring Contrast vs Spatial Frequency')
ax.set_ylim(0, 1.0)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[2]
for name, data in ring_data.items():
    radii, contrast = data['radii'], data['contrast']
    if len(radii) == 0: continue
    col = COLORS.get(name, 'gray')
    ax.scatter(radii, contrast, s=10, alpha=0.3, color=col)
    bin_edges = np.arange(RING_R_MIN, RING_R_MAX, 8)
    plot_binned(ax, radii, contrast, col, name, bin_edges)

ax.axhline(0.1, ls='--', color='orange', alpha=0.5)
ax.set_xlabel('Radius from centre (px)')
ax.set_ylabel('Michelson contrast')
ax.set_title('Ring Contrast vs Radius')
ax.set_ylim(0, 1.0)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Radial Ring Contrast Analysis (wedge {RING_ANGLE_MIN}°–{RING_ANGLE_MAX}°)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()